In [1]:
!nvidia-smi

Mon Jul  6 16:34:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.52                 Driver Version: 596.52         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A2000 Laptop GPU  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   45C    P0             16W /   65W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

2.5.1+cu121
True
NVIDIA RTX A2000 Laptop GPU


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
import os
from roboflow import Roboflow

api_key = os.getenv("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=api_key)

project = rf.workspace("arin-alexia").project("traffic-and-road-signs-2m6fa")
version = project.version(1)
dataset = version.download("yolov11")

data_yaml = os.path.join(dataset.location, 'data.yaml')
print('Dataset location:', dataset.location)

loading Roboflow workspace...
loading Roboflow project...
Dataset location: c:\Personal_Projects\traffic-sign-detection-project1\notebooks\Traffic-and-Road-Signs-1


In [5]:
import os

print('Files in dataset folder:')
for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for f in files[:10]:
        print(f'{subindent}{f}')

Files in dataset folder:
Traffic-and-Road-Signs-1/
  data.yaml
  README.dataset.txt
  README.roboflow.txt
  test/
    images/
      00000_00000_00000_png_jpg.rf.5c06efd0829a64988da46f03d0268f61.jpg
      00000_00000_00009_png_jpg.rf.fcff6c83c74508ba45f64a104d67b8fb.jpg
      00000_00000_00014_png_jpg.rf.91ce606aaa3e73f4bc65f4a3574efe2b.jpg
      00000_00000_00015_png_jpg.rf.9dce14d1ad236ae20f4b9b842be2f411.jpg
      00000_00000_00016_png_jpg.rf.896906129412dab5173dc9196494b865.jpg
      00000_00000_00017_png_jpg.rf.069a839d1720fa8d2da6e82b5add92c4.jpg
      00000_00000_00018_png_jpg.rf.5b94826df59c65c446684f4bf0c9f890.jpg
      00000_00000_00019_png_jpg.rf.9eda9d9735d037093df91665bf59ad0a.jpg
      00000_00000_00020_png_jpg.rf.045df81719c373af370b1238a3063bb3.jpg
      00000_00000_00021_png_jpg.rf.44292d978ea6d384f3d1469d1d96252a.jpg
    labels/
      00000_00000_00000_png_jpg.rf.5c06efd0829a64988da46f03d0268f61.txt
      00000_00000_00009_png_jpg.rf.fcff6c83c74508ba45f64a104d67b8fb.tx

In [6]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
print('Model loaded')

Model loaded


In [7]:
results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=-1,
    patience=15,
    device=0,

    mosaic=1.0,
    scale=0.5,
    copy_paste=0.1,

    hsv_h=0, hsv_s=0, hsv_v=0,
    fliplr=0, flipud=0,
    degrees=0, translate=0, shear=0, perspective=0, erasing=0,

    project='./outputs/metrics',
    name='v1d2_v1aug'
)

print(results)

New https://pypi.org/project/ultralytics/8.4.89 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.87  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A2000 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Personal_Projects\traffic-sign-detection-project1\notebooks\Traffic-and-Road-Signs-1\data.yaml, degrees=0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0, exist_ok=False, fliplr=0, flipud=0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, 

In [ ]:
# End of 7/5/2026 - training will suck (gonna be long as fuckkk), do it tomorrow, im done here, bye.
# 7/7/2026 - training is done, now to test the model and see how it performs on the validation set.

In [8]:
for root, dirs, files in os.walk('.'):
    if 'best.pt' in files:
        print(os.path.join(root, 'best.pt'))

.\runs\detect\outputs\metrics\v1d2_v1aug\weights\best.pt
.\runs\detect\outputs\metrics\v1d2_v1aug-2\weights\best.pt


In [ ]:
from ultralytics import YOLO
best_model_path = './runs/detect/outputs/metrics/v1d2_v1aug/weights/best.pt'
model = YOLO(best_model_path)

results = model.predict(
    source='C:/Personal_Projects/traffic-sign-detection-project1/notebooks/Traffic-and-Road-Signs-1/test/images/00000_00000_00000_png_jpg.rf.5c06efd0829a64988da46f03d0268f61.jpg',
    imgsz=640,
    conf=0.25,
    save=True,
    project='./outputs/predictions',
    name='sample_inference'
)

print(f"Detections saved to: {results[0].save_dir}")


image 1/1 C:\Personal_Projects\traffic-sign-detection-project1\notebooks\Traffic-and-Road-Signs-1\test\images\00000_00000_00000_png_jpg.rf.5c06efd0829a64988da46f03d0268f61.jpg: 640x640 1 Pedestrian Crossing, 1 Speed Limit 20 KMPh, 123.2ms
Speed: 11.5ms preprocess, 123.2ms inference, 16.1ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Personal_Projects\traffic-sign-detection-project1\notebooks\runs\detect\outputs\predictions\sample_inference
Detections saved to: C:\Personal_Projects\traffic-sign-detection-project1\notebooks\runs\detect\outputs\predictions\sample_inference


In [15]:
metrics = model.val(data=data_yaml)
print(metrics)

Ultralytics 8.4.87  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A2000 Laptop GPU, 4096MiB)


WARNING val: Slow image access detected (ping: 0.20.1 ms, read: 9.43.7 MB/s, size: 25.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning C:\Personal_Projects\traffic-sign-detection-project1\notebooks\Traffic-and-Road-Signs-1\valid\labels.cache... 1884 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1884/1884  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 118/118 2.1it/s 55.0s0.5s
                   all       1884       1886      0.935      0.946      0.941      0.798
-Road narrows on right         54         54      0.971          1      0.995      0.883
    50 mph speed limit         62         62      0.473          1      0.959      0.762
     Attention Please-        117        117      0.997          1      0.995      0.847
    Beware of children        106        106      0.996          1      0.995      

In [17]:
import json
import os
from datetime import datetime

results_summary = {
    "run_name": "v1d2_v1aug",
    "dataset": "Traffic-and-Road-Signs",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "model": "yolo11s",
    "epochs_configured": 100,
    "patience": 15,
    "augmentation": "mosaic=1.0, scale=0.5, copy_paste=0.1",
    "metrics": {
        "precision": round(metrics.results_dict['metrics/precision(B)'], 4),
        "recall": round(metrics.results_dict['metrics/recall(B)'], 4),
        "mAP50": round(metrics.results_dict['metrics/mAP50(B)'], 4),
        "mAP50-95": round(metrics.results_dict['metrics/mAP50-95(B)'], 4),
    }
}

print(json.dumps(results_summary, indent=2))

save_path = './outputs/metrics/v1d2_v1aug_summary.json'

# Ensure the parent directory exists before writing
os.makedirs(os.path.dirname(save_path), exist_ok=True)

with open(save_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\nSaved summary to: {save_path}")

{
  "run_name": "v1d2_v1aug",
  "dataset": "Traffic-and-Road-Signs",
  "timestamp": "2026-07-07 14:13",
  "model": "yolo11s",
  "epochs_configured": 100,
  "patience": 15,
  "augmentation": "mosaic=1.0, scale=0.5, copy_paste=0.1",
  "metrics": {
    "precision": 0.9347,
    "recall": 0.9461,
    "mAP50": 0.9408,
    "mAP50-95": 0.7976
  }
}

Saved summary to: ./outputs/metrics/v1d2_v1aug_summary.json


In [18]:
!powershell -Command "Get-ChildItem -Recurse -Filter best.pt -Path ./outputs"

In [20]:
import matplotlib.pyplot as plt
from PIL import Image
import os

print("Model classes:", model.names)

test_image_path = 'C:/Users/arina/Downloads/Project/stop.jpg'  # just point directly at a file on your computer

results = model.predict(
    source=test_image_path,
    imgsz=640,
    conf=0.25,
    save=True,
    project='./outputs/predictions',
    name='manual_upload_test'
)

print(f"Detections saved to: {results[0].save_dir}")

saved_files = os.listdir(results[0].save_dir)
annotated_path = os.path.join(results[0].save_dir, saved_files[0])
img = Image.open(annotated_path)

plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')
plt.show()

Model classes: {0: '-Road narrows on right', 1: '50 mph speed limit', 2: 'Attention Please-', 3: 'Beware of children', 4: 'CYCLE ROUTE AHEAD WARNING', 5: 'Dangerous Left Curve Ahead', 6: 'Dangerous Rright Curve Ahead', 7: 'End of all speed and passing limits', 8: 'Give Way', 9: 'Go Straight or Turn Right', 10: 'Go straight or turn left', 11: 'Keep-Left', 12: 'Keep-Right', 13: 'Left Zig Zag Traffic', 14: 'No Entry', 15: 'No_Over_Taking', 16: 'Overtaking by trucks is prohibited', 17: 'Pedestrian Crossing', 18: 'Round-About', 19: 'Slippery Road Ahead', 20: 'Speed Limit 20 KMPh', 21: 'Speed Limit 30 KMPh', 22: 'Stop_Sign', 23: 'Straight Ahead Only', 24: 'Traffic_signal', 25: 'Truck traffic is prohibited', 26: 'Turn left ahead', 27: 'Turn right ahead', 28: 'Uneven Road'}

image 1/1 C:\Users\arina\Downloads\Project\stop.jpg: 384x640 1 Stop_Sign, 103.0ms
Speed: 760.3ms preprocess, 103.0ms inference, 3.8ms postprocess per image at shape (1, 3, 384, 640)
Results saved to C:\Personal_Projects\tr

<Figure size 1000x1000 with 1 Axes>

In [24]:
results = model.predict(
    source=0,      # 0 = default webcam device
    imgsz=640,
    conf=0.65,
    show=True,     # opens a live window showing the annotated feed in real time
    save=False     # set True if you also want to save the video output to disk
)


1/1: 0... Success  (inf frames of shape 640x480 at 30.00 FPS)

WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

0: 480x640 (no detections), 30.2ms
0: 480x640 (no detections), 28.4ms
0: 480x640 (no detections), 29.2ms
0: 480x640 (no detections), 66.4ms
0: 480x640 (no detections), 22.0ms
0: 480x640 (no detections), 43.9ms
0: 480x640 1 Truck traffic is prohibited, 34.8ms
0: 480x640 1 Truck traffic is prohibited, 34.5ms
0: 480x640 (no detections), 16.6ms
0: 480x640 (no detections), 30.9ms
0: 480x640 (no detec